<a href="https://colab.research.google.com/github/Nariman-Mahmoud-Eldaly/RoadEye_Graduation_Project/blob/main/Road_Eye_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!fuser -k 8000/tcp

In [ ]:
!pip install fastapi uvicorn nest_asyncio pyngrok

In [ ]:
from pyngrok import ngrok
import nest_asyncio
import uvicorn


ngrok.set_auth_token("3FkaisKhSFTLPTLM2GszCBMB7BC_2Bf8Ka8pNEQCjfqJ65jRq")

In [ ]:
# ── 0. Mount Drive ────────────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# ── 1. Install Dependencies ───────────────────────────────────────────────────
import subprocess, sys

def pip_install(*pkgs):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *pkgs])

pip_install("filterpy")
pip_install("--upgrade", "ultralytics")
pip_install("opencv-python", "numpy", "torch", "torchvision", "requests")
pip_install("gradio")

In [ ]:
# ── 2. Imports ────────────────────────────────────────────────────────────────
import os
import cv2
import glob
import random
import time
from collections import defaultdict, deque

import gradio as gr
import numpy as np
import tensorflow as tf
import torch
from filterpy.kalman import KalmanFilter
from ultralytics import YOLO

# Colab display helper — only used during interactive debugging
try:
    from google.colab.patches import cv2_imshow   # noqa: F401
except ImportError:
    pass

In [ ]:
# ── 3. Constants ──────────────────────────────────────────────────────────────
PIXEL_TO_METER = 0.025

##4. Load Models

In [ ]:
# ── 4a. Driver-state classifier (EfficientNet) ────────────────────────────────
DRIVER_MODEL_PATH = "/content/drive/MyDrive/graduation project/RoadEye_best_model.keras"
driver_model = tf.keras.models.load_model(DRIVER_MODEL_PATH)
print("Driver model loaded ")

DRIVER_IMAGE_DIR = "/content/drive/MyDrive/graduation project/driver_test/*.jpg"
driver_image_paths = glob.glob(DRIVER_IMAGE_DIR)
print(f"Found {len(driver_image_paths)} driver images")

CLASS_NAMES_DRIVER = {
    0: "DangerousDriving",
    1: "Distracted",
    2: "Drinking",
    3: "SafeDriving",
    4: "SleepyDriving",
    5: "Yawn",
}

UNSAFE_DRIVER_STATES = {"DangerousDriving", "Distracted", "SleepyDriving", "Drinking"}


def predict_driver_state(frame: np.ndarray) -> tuple[str, float]:
    """Return (state_label, confidence) from a BGR driver frame."""
    img = cv2.resize(frame, (224, 224))
    img = tf.keras.applications.efficientnet.preprocess_input(img.astype(np.float32))
    img = np.expand_dims(img, axis=0)
    pred = driver_model.predict(img, verbose=0)
    cls = int(np.argmax(pred))
    return CLASS_NAMES_DRIVER[cls], float(pred[0][cls])

Driver model loaded 
Found 5 driver images


In [ ]:
# ── 4b. Fine-tuned YOLO ───────────────────────────────────────────────────────
TRAINED_WEIGHTS  = "/content/drive/MyDrive/graduation project/best.pt"
DEFAULT_WEIGHTS  = "yolov8n.pt"
DATASET_YAML     = "/content/dataset/data.yaml"

if os.path.exists(TRAINED_WEIGHTS):
    print("Using saved trained weights")
    yolo_model = YOLO(TRAINED_WEIGHTS)
else:
    print("No trained weights found — training now …")
    # Download dataset
    os.makedirs("dataset", exist_ok=True)
    os.system('wget "https://app.roboflow.com/ds/vTLr1h8yIc?key=650Ec5JEnQ" -O dataset.zip')
    os.system("unzip dataset.zip -d dataset")

    yolo_model = YOLO(DEFAULT_WEIGHTS)
    yolo_model.train(data=DATASET_YAML, epochs=10, imgsz=640)

    trained_path = "runs/detect/train/weights/best.pt"
    os.makedirs(os.path.dirname(TRAINED_WEIGHTS), exist_ok=True)
    os.rename(trained_path, TRAINED_WEIGHTS)
    yolo_model = YOLO(TRAINED_WEIGHTS)

CLASS_NAMES = yolo_model.names

# Identify vehicle and accident class IDs from the model
VEHICLE_KEYWORDS  = {"car", "truck", "bus", "motorcycle", "bicycle", "vehicle", "ambulance"}
ACCIDENT_KEYWORDS = {"accident", "crash", "collision"}

VEHICLE_CLASSES:  set[int] = set()
ACCIDENT_CLASSES: set[int] = set()

for _idx, _name in CLASS_NAMES.items():
    _n = _name.lower()
    if any(k in _n for k in VEHICLE_KEYWORDS):  VEHICLE_CLASSES.add(_idx)
    if any(k in _n for k in ACCIDENT_KEYWORDS): ACCIDENT_CLASSES.add(_idx)

# Fallback to standard COCO vehicle IDs if fine-tuned model has none
if not VEHICLE_CLASSES:
    VEHICLE_CLASSES = {2, 3, 5, 7}   # car, motorcycle, bus, truck

print(f"Vehicle class IDs : {VEHICLE_CLASSES}")
print(f"Accident class IDs: {ACCIDENT_CLASSES}")

Using saved trained weights
Vehicle class IDs : {3, 16, 18, 19, 20, 21, 22, 23, 24, 25, 26, 34, 35, 45, 46, 50, 51, 52, 53, 54}
Accident class IDs: {0, 17, 26, 6}


In [ ]:
# ── 4c. MiDaS depth model ────────────────────────────────────────────────────
print("Loading MiDaS depth model …")
midas           = torch.hub.load("intel-isl/MiDaS", "MiDaS_small")
midas.eval()
midas_transform = torch.hub.load("intel-isl/MiDaS", "transforms").small_transform
print("MiDaS loaded")

Loading MiDaS depth model …


Using cache found in /root/.cache/torch/hub/intel-isl_MiDaS_master


Loading weights:  None


Using cache found in /root/.cache/torch/hub/rwightman_gen-efficientnet-pytorch_master


MiDaS loaded


Using cache found in /root/.cache/torch/hub/intel-isl_MiDaS_master


## 5. Utility Functions

In [ ]:
def bbox_iou(b1: tuple, b2: tuple) -> float:
    """Intersection-over-Union of two (x1,y1,x2,y2) boxes."""
    x1,  y1,  x2,  y2  = b1
    x1g, y1g, x2g, y2g = b2
    ix1, iy1 = max(x1, x1g), max(y1, y1g)
    ix2, iy2 = min(x2, x2g), min(y2, y2g)
    iw = max(ix2 - ix1, 0)
    ih = max(iy2 - iy1, 0)
    inter = iw * ih
    union = (x2-x1)*(y2-y1) + (x2g-x1g)*(y2g-y1g) - inter
    return inter / union if union > 0 else 0.0

In [ ]:
def remove_duplicates(
    detections: list[dict],
    iou_thresh: float = 0.3,
    center_dist_thresh: float = 80,
) -> list[dict]:
    """
    Remove duplicate bounding boxes for the same physical object.
    Keeps the highest-confidence detection when two boxes overlap
    (by IoU OR centroid distance).
    """
    filtered: list[dict] = []
    for det in sorted(detections, key=lambda x: x["conf"], reverse=True):
        x1, y1, x2, y2 = det["bbox"]
        cx = (x1 + x2) / 2
        cy = (y1 + y2) / 2
        keep = True
        for f in filtered:
            fx1, fy1, fx2, fy2 = f["bbox"]
            fcx = (fx1 + fx2) / 2
            fcy = (fy1 + fy2) / 2
            # Class is intentionally ignored — same spatial location = same object
            if (bbox_iou(det["bbox"], f["bbox"]) > iou_thresh or
                    np.hypot(cx - fcx, cy - fcy) < center_dist_thresh):
                keep = False
                break
        if keep:
            filtered.append(det)
    return filtered

In [ ]:
def is_valid_vehicle_box(x1: float, y1: float, x2: float, y2: float,
                          frame_w: int, frame_h: int,
                          min_area: int = 500,
                          aspect_lo: float = 0.3,
                          aspect_hi: float = 5.0) -> bool:
    """
    Returns True only if a detection looks like a real vehicle bounding box.

    Filters that reduce false positives
    ────────────────────────────────────
    1. Aspect ratio — very tall/thin or very wide/flat boxes are unlikely vehicles.
    2. Minimum area  — tiny blobs (noise, shadows) are rejected.
    3. Edge clipping — a box completely against a frame edge is probably a crop artifact.
    """
    w = x2 - x1
    h = y2 - y1
    if h <= 0 or w <= 0:
        return False
    aspect = w / h
    if not (aspect_lo <= aspect <= aspect_hi):
        return False
    if w * h < min_area:
        return False
    # Reject boxes that are entirely at the very edge of the frame (likely artifacts)
    margin = 3
    if x2 <= margin or y2 <= margin or x1 >= frame_w - margin or y1 >= frame_h - margin:
        return False
    return True

 ## 6. Core Classes

In [ ]:
class SimpleTracker:
    """
    Centroid-based multi-object tracker with depth support.

    Matching strategy: score = 0.7 * IoU  +  0.3 * proximity_score
    A match is accepted only when score > 0.5 AND centroid distance < max_distance.
    """

    def __init__(self, max_distance: int = 80, max_lost: int = 30):
        self.next_id      = 1
        self.tracks: dict = {}
        self.max_distance = max_distance
        self.max_lost     = max_lost

    def update(self, detections: list[dict], class_names_map: dict) -> list[dict]:
        updated_tracks: dict = {}
        used_dets: set[int]  = set()

        det_centers = [
            ((d["bbox"][0] + d["bbox"][2]) / 2, (d["bbox"][1] + d["bbox"][3]) / 2)
            for d in detections
        ]

        # ── Match existing tracks ────────────────────────────────────────────
        for track_id, track in self.tracks.items():
            last_cx, last_cy = track["center"]
            best_idx, best_score = None, -1.0

            for i, (cx, cy) in enumerate(det_centers):
                if i in used_dets:
                    continue
                dist = np.hypot(cx - last_cx, cy - last_cy)
                if dist >= self.max_distance:
                    continue   # too far — disqualify early
                iou   = bbox_iou(track["bbox"], detections[i]["bbox"])
                score = 0.7 * iou + 0.3 * (1.0 - dist / self.max_distance)
                if score > best_score:
                    best_score = score
                    best_idx   = i

            if best_idx is not None and best_score > 0.5:
                det    = detections[best_idx]
                cx, cy = det_centers[best_idx]
                prev_cx, prev_cy = track["center"]
                used_dets.add(best_idx)
                updated_tracks[track_id] = {
                    "center"           : (cx, cy),
                    "prev_center"      : (prev_cx, prev_cy),
                    "velocity"         : np.array([cx - prev_cx, cy - prev_cy]),
                    "depth"            : track.get("depth"),
                    "bbox"             : det["bbox"],
                    "cls"              : det["cls"],
                    "class_name"       : class_names_map.get(det["cls"], "vehicle"),
                    "conf"             : det["conf"],
                    "mask"             : det.get("mask"),
                    "lost"             : 0,
                    "is_accident_class": det.get("is_accident_class", False),
                }
            else:
                track["lost"] += 1
                if track["lost"] <= self.max_lost:
                    updated_tracks[track_id] = track

        # ── Create new tracks for unmatched detections ───────────────────────
        for i, det in enumerate(detections):
            if i in used_dets:
                continue
            x1, y1, x2, y2 = det["bbox"]
            cx = (x1 + x2) / 2
            cy = (y1 + y2) / 2
            tid = self.next_id
            self.next_id += 1
            updated_tracks[tid] = {
                "center"           : (cx, cy),
                "prev_center"      : (cx, cy),
                "velocity"         : np.array([0.0, 0.0]),
                "depth"            : None,
                "bbox"             : det["bbox"],
                "cls"              : det["cls"],
                "class_name"       : class_names_map.get(det["cls"], "vehicle"),
                "conf"             : det["conf"],
                "mask"             : det.get("mask"),
                "lost"             : 0,
                "is_accident_class": det.get("is_accident_class", False),
            }

        self.tracks = updated_tracks
        return [{"id": tid, **t} for tid, t in self.tracks.items()]

In [ ]:
class EnhancedMotionAnalyzer:
    """
    Velocity / acceleration / depth analysis.
    Converging-path check + sudden-stop detection.
    """

    def __init__(self, fps: int = 30, history_len: int = 15):
        self.fps        = fps
        self.history    = defaultdict(lambda: deque(maxlen=history_len))
        self.depth_hist = defaultdict(lambda: deque(maxlen=history_len))
        self.vel_hist   = defaultdict(lambda: deque(maxlen=history_len))

    def update(self, tracks: list[dict]) -> None:
        for t in tracks:
            tid  = t["id"]
            cx, cy = t["center"]
            self.history[tid].append((cx, cy))
            if t.get("depth") is not None:
                self.depth_hist[tid].append(t["depth"])
            if len(self.history[tid]) >= 2:
                h  = list(self.history[tid])
                vx = h[-1][0] - h[-2][0]
                vy = h[-1][1] - h[-2][1]
                self.vel_hist[tid].append((vx, vy))

    # ── velocity ─────────────────────────────────────────────────────────────

    def get_velocity(self, tid: int) -> tuple[float, float]:
        h = self.history[tid]
        if len(h) < 2:
            return 0.0, 0.0
        (x1, y1), (x2, y2) = list(h)[-2:]
        return x2 - x1, y2 - y1

    def get_smoothed_velocity(self, tid: int, window: int = 5) -> tuple[float, float]:
        h = list(self.history[tid])
        if len(h) < 2:
            return 0.0, 0.0
        vxs, vys = [], []
        for i in range(1, min(len(h), window + 1)):
            vxs.append(h[-i][0] - h[-i - 1][0])
            vys.append(h[-i][1] - h[-i - 1][1])
        return float(np.mean(vxs)), float(np.mean(vys))

    def get_speed(self, tid: int) -> float:
        return float(np.hypot(*self.get_velocity(tid)))

    def get_smoothed_speed(self, tid: int, window: int = 5) -> float:
        return float(np.hypot(*self.get_smoothed_velocity(tid, window)) * self.fps)

    def get_speed_kmh(self, tid: int) -> float:
        speed = self.get_speed(tid) * self.fps * PIXEL_TO_METER * 3.6
        return speed if 0 <= speed <= 250 else 0.0

    # ── direction ─────────────────────────────────────────────────────────────

    def get_direction_label(self, tid: int) -> str:
        """Returns a unicode arrow symbol for the direction of motion."""
        vx, vy = self.get_velocity(tid)
        deg    = (np.degrees(np.arctan2(vy, vx)) + 360) % 360
        arrows = ["→", "↗", "↑", "↖", "←", "↙", "↓", "↘"]
        return arrows[int((deg + 22.5) / 45) % 8]

    # ── acceleration ──────────────────────────────────────────────────────────

    def get_deceleration(self, tid: int) -> float:
        h = list(self.history[tid])
        if len(h) < 3:
            return 0.0
        v_prev = np.hypot(h[-2][0]-h[-3][0], h[-2][1]-h[-3][1]) * self.fps
        v_now  = np.hypot(h[-1][0]-h[-2][0], h[-1][1]-h[-2][1]) * self.fps
        return float(v_prev - v_now)

    def get_acceleration_kmh2(self, tid: int) -> float:
        return -self.get_deceleration(tid) * PIXEL_TO_METER * 3.6

    # ── depth ─────────────────────────────────────────────────────────────────

    def get_depth(self, tid: int) -> float:
        d = self.depth_hist[tid]
        return float(d[-1]) if d else 0.0

    def get_smoothed_depth(self, tid: int, window: int = 5) -> float:
        d = list(self.depth_hist[tid])
        return float(np.mean(d[-window:])) if d else 0.0

    # ── prediction ───────────────────────────────────────────────────────────

    def _get_acceleration_vector(self, tid: int) -> tuple[float, float]:
        vh = list(self.vel_hist[tid])
        if len(vh) >= 2:
            return vh[-1][0] - vh[-2][0], vh[-1][1] - vh[-2][1]
        return 0.0, 0.0

    def predict_future_center(self, tid: int, n_seconds: float):
        h = list(self.history[tid])
        if not h:
            return None
        cx, cy   = h[-1]
        vx, vy   = self.get_smoothed_velocity(tid)
        ax, ay   = self._get_acceleration_vector(tid)
        steps    = n_seconds * self.fps
        return (cx + vx*steps + 0.5*ax*steps**2,
                cy + vy*steps + 0.5*ay*steps**2)

    def predict_future_bbox(self, track: dict, n_seconds: float,
                            use_acceleration: bool = True) -> tuple:
        tid        = track["id"]
        x1,y1,x2,y2 = track["bbox"]
        w, h         = x2-x1, y2-y1
        cx, cy       = track["center"]
        vx, vy       = self.get_smoothed_velocity(tid)
        steps        = n_seconds * self.fps
        ax, ay       = self._get_acceleration_vector(tid) if use_acceleration else (0.0, 0.0)
        fx = cx + vx*steps + 0.5*ax*steps**2
        fy = cy + vy*steps + 0.5*ay*steps**2
        return (fx - w/2, fy - h/2, fx + w/2, fy + h/2)

    def get_future_trajectory_points(self, tid: int, current_center: tuple,
                                     n_seconds: float = 3, n_points: int = 8) -> list:
        vx, vy   = self.get_smoothed_velocity(tid)
        ax, ay   = self._get_acceleration_vector(tid)
        cx, cy   = current_center
        points   = [(cx, cy)]
        for i in range(1, n_points + 1):
            t     = (n_seconds / n_points) * i
            steps = t * self.fps
            points.append((cx + vx*steps + 0.5*ax*steps**2,
                           cy + vy*steps + 0.5*ay*steps**2))
        return points

    # ── TTC & distance ────────────────────────────────────────────────────────

    def estimate_ttc(self, t1: dict, t2: dict):
        """
        Time-To-Collision (seconds) using relative velocity projection.
        Returns None if vehicles are not converging.
        """
        pos1 = np.array(t1["center"])
        pos2 = np.array(t2["center"])
        vx1, vy1 = self.get_smoothed_velocity(t1["id"])
        vx2, vy2 = self.get_smoothed_velocity(t2["id"])
        rel_vel  = np.array([vx1 - vx2, vy1 - vy2])
        rel_pos  = pos2 - pos1
        dist     = np.linalg.norm(rel_pos)
        if dist < 1:
            return 0.0
        closing_speed = np.dot(rel_vel, rel_pos / dist)
        if closing_speed <= 0.5:
            return None
        return round(dist / closing_speed / self.fps, 1)

    def distance_between(self, t1: dict, t2: dict) -> float:
        cx1, cy1 = t1["center"]
        cx2, cy2 = t2["center"]
        return float(np.hypot(cx2 - cx1, cy2 - cy1))

    def are_converging(self, tid1: int, tid2: int,
                       threshold: float = 0.3,
                       min_closing_speed: float = 0.8) -> bool:

        h1 = list(self.history[tid1])
        h2 = list(self.history[tid2])
        if len(h1) < 3 or len(h2) < 3:
            return False

        pos1 = np.array(h1[-1])
        pos2 = np.array(h2[-1])
        dir1 = pos1 - np.array(h1[-3])
        dir2 = pos2 - np.array(h2[-3])
        n1, n2 = np.linalg.norm(dir1), np.linalg.norm(dir2)

        to_other = pos2 - pos1
        to_n     = np.linalg.norm(to_other)
        if to_n < 1:
            return False
        to_other_norm = to_other / to_n

        vx1, vy1 = self.get_smoothed_velocity(tid1)
        vx2, vy2 = self.get_smoothed_velocity(tid2)
        rel_vel        = np.array([vx1 - vx2, vy1 - vy2])
        closing_speed  = float(np.dot(rel_vel, to_other_norm))

        # Mutual convergence (normal pre-collision)
        if n1 >= 1 and n2 >= 1:
            d1 = dir1 / n1
            d2 = dir2 / n2
            if (np.dot(d1,  to_other_norm) > threshold and
                    np.dot(d2, -to_other_norm) > threshold and
                    closing_speed > min_closing_speed):
                return True

        # One-sided: one vehicle moving toward a slow/stopped other
        if n1 >= 1 and np.dot(dir1 / n1, to_other_norm) > threshold and closing_speed > min_closing_speed:
            return True
        if n2 >= 1 and np.dot(dir2 / n2, -to_other_norm) > threshold and closing_speed > min_closing_speed:
            return True

        return False

    def had_sudden_stop(self, tid: int,
                        window: int = 5,
                        speed_before: float = 10.0,
                        speed_after:  float = 2.0) -> bool:
        """True if a vehicle dropped from high speed to near-stop recently."""
        h = list(self.history[tid])
        if len(h) < window + 1:
            return False
        speeds = [np.hypot(h[i][0]-h[i-1][0], h[i][1]-h[i-1][1])
                  for i in range(1, len(h))]
        recent = speeds[-window:]
        return len(recent) >= 3 and recent[0] > speed_before and recent[-1] < speed_after

 ## 7. Future-collision prediction helpers

In [ ]:
def build_future_bbox_matrix(tracks: list[dict],
                              motion: EnhancedMotionAnalyzer,
                              horizons: list[float]) -> np.ndarray:
    """Returns array of shape [N_tracks, N_horizons, 4]."""
    matrix = np.zeros((len(tracks), len(horizons), 4), dtype=np.float32)
    for i, t in enumerate(tracks):
        for h_i, sec in enumerate(horizons):
            matrix[i, h_i] = motion.predict_future_bbox(t, sec, use_acceleration=True)
    return matrix

In [ ]:
def predict_collision_windows(tracks: list[dict],
                               bbox_matrix: np.ndarray,
                               horizons: list[float],
                               motion: EnhancedMotionAnalyzer,
                               iou_thresh: float = 0.15) -> list[tuple]:
    windows = []
    for i in range(len(tracks)):
        for j in range(i + 1, len(tracks)):
            # Skip if boxes already overlap heavily → same physical object
            if bbox_iou(tracks[i]["bbox"], tracks[j]["bbox"]) > 0.25:
                continue
            # Skip if centroids are already very close → same physical object
            cx1, cy1 = tracks[i]["center"]
            cx2, cy2 = tracks[j]["center"]
            if np.hypot(cx1 - cx2, cy1 - cy2) < 60:
                continue

            times = [
                sec for h_i, sec in enumerate(horizons)
                if bbox_iou(bbox_matrix[i, h_i], bbox_matrix[j, h_i]) > iou_thresh
            ]
            if times:
                windows.append((tracks[i]["id"], tracks[j]["id"],
                                 min(times), max(times)))
    return windows

 ## 8. Accident Severity

In [ ]:
# ── 8. Accident Severity ──────────────────────────────────────────────────────

def accident_severity(t1: dict, t2: dict, motion: EnhancedMotionAnalyzer) -> str:
    speed1 = motion.get_speed(t1["id"])
    speed2 = motion.get_speed(t2["id"])
    dist   = motion.distance_between(t1, t2)
    iou    = bbox_iou(t1["bbox"], t2["bbox"])

    if iou > 0.4 or (speed1 > 5 and speed2 > 5 and dist < 50):
        return "SEVERE"
    if iou > 0.2 or (speed1 > 2 and speed2 > 2 and dist < 80):
        return "MEDIUM"
    return "MINOR"

## 9. Hybrid Accident Detecto

In [ ]:
class HybridAccidentDetector:
    """
    Combines:
    - IoU overlap + depth proximity
    - Converging trajectories + closing speed
    - Sudden stop detection
    - Direct accident-class signal from fine-tuned YOLO
    - Temporal confirmation (≥3 consecutive high-risk frames)

    False-positive mitigations
    ──────────────────────────
    • Pairs with large IoU (>0.25) are treated as the same vehicle and skipped.
    • Non-converging pairs score 0 unless an explicit accident class is detected.
    • Risk score is smoothed over a 5-frame window; a single spike cannot confirm.
    """

    def __init__(
        self,
        iou_thresh:          float = 0.1,
        min_distance_3d:     float = 400.0,
        min_frames_confirm:  int   = 3,
        risk_thresh:         float = 0.45,
    ):
        self.iou_thresh         = iou_thresh
        self.min_distance_3d    = min_distance_3d
        self.min_frames_confirm = min_frames_confirm
        self.risk_thresh        = risk_thresh

        self.collision_frames   = 0
        self.in_accident        = False
        self.warning_history:   dict[tuple, list[float]] = defaultdict(list)
        self.accident_log:      list[dict] = []

    def _score_pair(self, t1: dict, t2: dict,
                motion: EnhancedMotionAnalyzer) -> tuple[float, list[str]]:

        # Guard: same physical object (heavy IoU overlap)
        if bbox_iou(t1["bbox"], t2["bbox"]) > 0.25:
            return 0.0, ["Same vehicle (IoU overlap)"]

        tid1, tid2 = t1["id"], t2["id"]
        dist_px    = motion.distance_between(t1, t2)

        # Guard: too far apart (pixel distance; min_distance_3d is now in pixels)
        if dist_px > self.min_distance_3d:
            return 0.0, []

        risk:    float      = 0.0
        factors: list[str]  = []

        is_converging   = motion.are_converging(tid1, tid2)
        iou             = bbox_iou(t1["bbox"], t2["bbox"])
        speed1          = motion.get_smoothed_speed(tid1)
        speed2          = motion.get_smoothed_speed(tid2)
        has_sudden_stop = motion.had_sudden_stop(tid1) or motion.had_sudden_stop(tid2)
        is_accident_cls = t1.get("is_accident_class") or t2.get("is_accident_class")

        # ── Path A: Converging (pre-collision) ───────────────────────────────
        if is_converging:
            risk += 0.35
            factors.append("Converging paths")

            px_dist = dist_px
            if px_dist < 150:
                risk += (150 - px_dist) / 150 * 0.3
                factors.append(f"Close: {px_dist:.0f}px")

            rel_speed = abs(speed1 - speed2)
            if rel_speed > 5:
                risk += min(rel_speed / 20, 0.2)
                factors.append(f"Rel speed: {rel_speed:.1f}px/s")

        # ── Path B: Overlapping boxes + at least one moving (collision now) ──
        if iou > self.iou_thresh and (speed1 >= 2 or speed2 >= 2):
            risk += iou * 0.55
            factors.append(f"IoU overlap: {iou:.2f}")

        # ── Path C: Explicit accident class from YOLO ────────────────────────
        if is_accident_cls:
            risk += 0.25
            factors.append("Accident class signal")

        # ── Path D: Sudden stop near another vehicle ─────────────────────────
        if has_sudden_stop:
            risk += 0.2
            factors.append("Sudden stop")

        # ── Path E: Static proximity — both vehicles very slow/stopped and close
        both_slow = speed1 < 3.0 and speed2 < 3.0
        if both_slow and dist_px < 120:
            depth1 = motion.get_smoothed_depth(tid1)
            depth2 = motion.get_smoothed_depth(tid2)
            depth_diff = abs(depth1 - depth2)
            # Low depth difference → same depth plane → same road → suspicious
            if depth_diff < 0.15:
                risk += 0.35
                factors.append(f"Static proximity: {dist_px:.0f}px same-plane")

        # Nothing triggered at all
        if not factors:
            return 0.0, ["No collision signal"]

        return min(risk, 1.0), factors

    def update(self, tracks: list[dict],
               motion: EnhancedMotionAnalyzer,
               frame_count: int) -> tuple[list, bool, str]:

        high_risk_pairs:        list = []
        accident_pair_detected: bool = False
        worst_severity:         str  = "MINOR"

        accident_class_tracks = [t for t in tracks if t.get("is_accident_class")]
        if accident_class_tracks:
            key = ("accident_cls", frame_count // 5)   # bucket by 5-frame windows
            self.warning_history[key].append(0.9)
            if len(self.warning_history[key]) > 5:
                self.warning_history[key].pop(0)
            recent = self.warning_history[key]
            if len(recent) >= 2:   # only 2 frames needed — model is reliable
                accident_pair_detected = True
                worst_severity = "SEVERE"
                # Highlight the accident-class track in high_risk list
                for t in accident_class_tracks:
                    high_risk_pairs.append(
                        (t["id"], t["id"], 0.9, ["YOLO accident class"])
                    )

        for i in range(len(tracks)):
            for j in range(i + 1, len(tracks)):
                t1, t2 = tracks[i], tracks[j]
                risk, factors = self._score_pair(t1, t2, motion)

                key = (t1["id"], t2["id"])
                self.warning_history[key].append(risk)
                if len(self.warning_history[key]) > 5:
                    self.warning_history[key].pop(0)

                if risk > self.risk_thresh:
                    high_risk_pairs.append((t1["id"], t2["id"], risk, factors))

                # Temporal confirmation: mean of last 3 frames above threshold
                recent = self.warning_history[key]
                if len(recent) >= 3 and float(np.mean(recent[-3:])) > self.risk_thresh:
                    accident_pair_detected = True
                    sev = accident_severity(t1, t2, motion)
                    if sev == "SEVERE" or worst_severity == "MINOR":
                        worst_severity = sev

        # Update collision frame counter (decay when no pair detected)
        if accident_pair_detected:
            self.collision_frames += 1
        else:
            self.collision_frames = max(0, self.collision_frames - 1)

        accident_confirmed = False
        if self.collision_frames >= self.min_frames_confirm and not self.in_accident:
            self.in_accident   = True
            accident_confirmed = True
            self.accident_log.append({"frame": frame_count, "severity": worst_severity})
            print(f"ACCIDENT CONFIRMED at frame {frame_count} — {worst_severity}")

        if self.collision_frames == 0:
            self.in_accident = False

        return high_risk_pairs, accident_confirmed, worst_severity

## 10. Drawing helpers

In [ ]:
def draw_direction_arrow(frame: np.ndarray, center: tuple, velocity: tuple,
                         color: tuple, scale: float = 3.0,
                         min_speed: float = 0.5) -> None:
    vx, vy = velocity
    speed  = np.hypot(vx, vy)
    if speed < min_speed:
        return
    cx, cy     = int(center[0]), int(center[1])
    arrow_len  = min(speed * scale, 60)
    ex = int(cx + (vx / speed) * arrow_len)
    ey = int(cy + (vy / speed) * arrow_len)
    cv2.arrowedLine(frame, (cx, cy), (ex, ey), color, 2, tipLength=0.35)

In [ ]:
def draw_future_trajectory(frame: np.ndarray, traj_points: list,
                           color: tuple, danger: bool = False) -> None:
    if len(traj_points) < 2:
        return
    dash_color = (0, 80, 255) if danger else color
    n          = len(traj_points) - 1
    for i in range(n):
        p1    = (int(traj_points[i][0]),     int(traj_points[i][1]))
        p2    = (int(traj_points[i+1][0]),   int(traj_points[i+1][1]))
        alpha = 0.9 * (1 - i / n)
        seg_color = tuple(int(c * alpha) for c in dash_color)
        thickness = 2 if danger else 1
        if i % 2 == 0:
            cv2.line(frame, p1, p2, seg_color, thickness, cv2.LINE_AA)
        dot_r = 3 if i == n - 1 else 2
        cv2.circle(frame, p2, dot_r, seg_color, -1, cv2.LINE_AA)

In [ ]:
def draw_predicted_collision_zone(frame: np.ndarray,
                                   future_pt1, future_pt2,
                                   ttc) -> None:
    if future_pt1 is None or future_pt2 is None:
        return
    mx = int((future_pt1[0] + future_pt2[0]) / 2)
    my = int((future_pt1[1] + future_pt2[1]) / 2)

    if ttc is not None and ttc < 1.0:
        zone_color = (0,   0, 255)
    elif ttc is not None and ttc < 2.0:
        zone_color = (0, 128, 255)
    else:
        zone_color = (0, 220, 255)

    cv2.circle(frame, (mx, my), 28, zone_color, 2, cv2.LINE_AA)
    cv2.circle(frame, (mx, my), 18, zone_color, 1, cv2.LINE_AA)
    cv2.drawMarker(frame, (mx, my), zone_color,
                   cv2.MARKER_TILTED_CROSS, 14, 2, cv2.LINE_AA)
    if ttc is not None:
        cv2.putText(frame, f"TTC:{ttc:.1f}s", (mx - 28, my - 34),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, zone_color, 1, cv2.LINE_AA)

In [ ]:
def draw_risk_meter(frame: np.ndarray, risk_score: float,
                    x: int, y: int, width: int = 120,
                    height: int = 12, label: str = "RISK") -> None:
    cv2.rectangle(frame, (x, y), (x + width, y + height), (50, 50, 50), -1)
    cv2.rectangle(frame, (x, y), (x + width, y + height), (150, 150, 150), 1)
    filled_w = int(width * min(risk_score, 1.0))
    if filled_w > 0:
        if risk_score < 0.5:
            g, r = 255, int(risk_score * 2 * 255)
        else:
            r, g = 255, int((1 - risk_score) * 2 * 255)
        cv2.rectangle(frame, (x, y), (x + filled_w, y + height), (0, g, r), -1)
    cv2.putText(frame, f"{label} {risk_score:.0%}",
                (x, y - 3), cv2.FONT_HERSHEY_SIMPLEX, 0.38, (220, 220, 220), 1)

In [ ]:
def draw_info_panel(frame: np.ndarray, t: dict,
                    motion: EnhancedMotionAnalyzer,
                    risk_score=None) -> None:
    tid         = t["id"]
    x1,y1,x2,y2 = [int(v) for v in t["bbox"]]
    speed_kmh   = motion.get_speed_kmh(tid)
    accel       = motion.get_acceleration_kmh2(tid)
    dir_sym     = motion.get_direction_label(tid)
    depth_val   = t.get("depth", 0)
    accel_sym   = "▲" if accel > 0.5 else ("▼" if accel < -0.5 else "━")
    label       = f"#{tid} {t['class_name']} {dir_sym} {speed_kmh:.0f}km/h {accel_sym}"

    cv2.putText(frame, label, (x1, y1 - 7),
                cv2.FONT_HERSHEY_SIMPLEX, 0.44, (180, 255, 180), 1, cv2.LINE_AA)
    cv2.putText(frame, f"d:{depth_val:.1f}",
                (x1, y2 + 12), cv2.FONT_HERSHEY_SIMPLEX, 0.35, (180, 180, 180), 1)

    if risk_score is not None and risk_score > 0.2:
        meter_w = max(x2 - x1, 60)
        draw_risk_meter(frame, risk_score, x1, y2 + 18, width=meter_w, height=8)

Preparing the GUI

## 11. Main Processing Loop

In [ ]:
OUTPUT_VIDEO_PATH = "/content/output_roadeye.mp4"
TRAJ_SECONDS      = 2.5
TRAJ_POINTS       = 10
TIME_HORIZONS     = [1, 2, 3, 5]   # prediction horizons in seconds
driver_confirmed_flag = {"value": False}
MAX_ATTEMPTS          = 3

alert work on real processing

In [ ]:
import time
import random

class RoadEyeState:
    def __init__(self, fps: int = 30):
        self.fps    = fps
        self.width  = None
        self.height = None

        self.tracker  = SimpleTracker(max_distance=80, max_lost=30)
        self.motion   = EnhancedMotionAnalyzer(fps=fps, history_len=15)
        self.detector = HybridAccidentDetector(
            iou_thresh=0.15,
            min_frames_confirm=3,
            risk_thresh=0.5,
        )

        self.accident_warning_count = 0
        self.driver_warning_count   = 0
        self.frame_count            = 0

        self.last_accident_alert_time     = -9999.0
        self.last_driver_alert_time       = -9999.0
        self.last_predictive_warning_time = -9999.0

        self.emergency_message     = "No Action Required"
        self.current_lat           = BASE_LAT
        self.current_lon           = BASE_LON
        self.collision_windows     = []
        self.id_colors             = {}
        self.track_risk_map        = {}
        self.driver_confirmed_flag = {"value": False}
        self.collision_frames = 0

        # يمنع إرسال Emergency أكثر من مرة
        self.emergency_sent = False

        self.driver_state   = "Normal"
        self.driver_img     = None
        self.driver_preview = None
        self._init_driver()

    def _init_driver(self):
        path                = random.choice(driver_image_paths)
        self.driver_img     = cv2.imread(path)
        self.driver_preview = cv2.resize(self.driver_img, (180, 180))
        self.driver_state, _ = predict_driver_state(self.driver_img)

    def reset_after_confirm(self):
        self.driver_warning_count   = 0
        self.accident_warning_count = 0
        self.driver_confirmed_flag["value"] = False
        # يسمح بإرسال Emergency مرة جديدة
        self.emergency_sent = False
        self.emergency_message = "Driver confirmed OK"

print("RoadEyeState defined OK")

RoadEyeState defined OK


In [ ]:
"""def process_frame(frame, state, lat=None, lon=None):
    state.frame_count += 1
    now = time.time()

    if state.width is None:
        state.height, state.width = frame.shape[:2]

    HOOD_LINE_Y = int(state.height * 0.8)

    # A. Driver
    if state.frame_count % (state.fps * 10) == 0 or state.frame_count == 1:
        state._init_driver()

    # B. MiDaS
    img_rgb     = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    input_batch = midas_transform(img_rgb)
    with torch.no_grad():
        depth_map = midas(input_batch).squeeze().cpu().numpy()

    # C. YOLO
    results    = yolo_model(frame, conf=0.3, iou=0.4, verbose=False)[0]
    has_masks  = results.masks is not None
    boxes_iter = (
        zip(results.boxes, results.masks.data) if has_masks
        else zip(results.boxes, [None] * len(results.boxes))
    )

    raw_detections = []
    for box, mask in boxes_iter:
        cls  = int(box.cls[0])
        conf = float(box.conf[0])
        x1, y1, x2, y2 = box.xyxy[0].cpu().numpy()
        is_vehicle  = cls in VEHICLE_CLASSES
        is_accident = cls in ACCIDENT_CLASSES
        if not (is_vehicle or is_accident):
            continue
        if not is_valid_vehicle_box(x1, y1, x2, y2, state.width, state.height):
            continue
        raw_detections.append({
            "bbox": (float(x1), float(y1), float(x2), float(y2)),
            "cls": cls, "conf": conf,
            "mask": mask.cpu().numpy() if mask is not None else None,
            "is_accident_class": is_accident,
        })

    detections = remove_duplicates(raw_detections, iou_thresh=0.3, center_dist_thresh=80)

    # D. Tracking
    tracks = state.tracker.update(detections, CLASS_NAMES)

    # E. Depth
    for t in tracks:
        if t.get("mask") is not None:
            resized_mask = cv2.resize(
                t["mask"].astype(np.uint8),
                (depth_map.shape[1], depth_map.shape[0]),
                interpolation=cv2.INTER_NEAREST,
            )
            mask_bool = resized_mask > 0
            if np.any(mask_bool):
                depth_val = float(np.median(depth_map[mask_bool]))
                state.tracker.tracks[t["id"]]["depth"] = depth_val
                t["depth"] = depth_val
        else:
            x1, y1, x2, y2 = [int(v) for v in t["bbox"]]
            cx_ = int(np.clip((x1+x2)//2, 0, depth_map.shape[1]-1))
            cy_ = int(np.clip((y1+y2)//2, 0, depth_map.shape[0]-1))
            depth_val = float(depth_map[cy_, cx_])
            state.tracker.tracks[t["id"]]["depth"] = depth_val
            t["depth"] = depth_val

    # F. Motion
    state.motion.update(tracks)

    # G. Collision prediction
    if state.frame_count % state.fps == 0 and len(tracks) >= 2:
        bbox_matrix             = build_future_bbox_matrix(tracks, state.motion, TIME_HORIZONS)
        state.collision_windows = predict_collision_windows(
            tracks, bbox_matrix, TIME_HORIZONS, state.motion
        )

    # H. Accident detection
    high_risk_pairs, accident_confirmed, severity_str = state.detector.update(
        tracks, state.motion, state.frame_count
    )

    # I. Alert logic
    system_status = "SAFE"
    hood_in_frame = any(int(t["bbox"][3]) >= HOOD_LINE_Y for t in tracks)

    if hood_in_frame and not state.driver_confirmed_flag["value"]:
        accident_confirmed = True
        severity_str       = "SEVERE"

    if state.driver_confirmed_flag["value"]:
        state.reset_after_confirm()
        system_status = "CONFIRMED SAFE"

    elif accident_confirmed or state.detector.in_accident:
        system_status = "ACCIDENT DETECTED"
        if (
            (now - state.last_accident_alert_time) >= ALERT_INTERVAL_SECS
            and not state.emergency_sent
        ):
            state.accident_warning_count  += 1
            state.last_accident_alert_time = now
            if state.accident_warning_count == 1:
                state.emergency_message = "ALERT 1 — Possible accident detected"
            elif state.accident_warning_count == 2:
                state.emergency_message = "ALERT 2 — Driver still unresponsive"
            elif state.accident_warning_count >= 3:
                state.emergency_sent = True
                if lat is not None: state.current_lat = lat
                if lon is not None: state.current_lon = lon
                state.emergency_message = (
                    f"ALERT 3 — EMERGENCY DISPATCHED\n"
                    f"Location: {state.current_lat}, {state.current_lon}"
                )

    elif state.driver_state in UNSAFE_DRIVER_STATES:
        # ← التعديل: لو emergency اتبعت، ثبّت الـ status على DRIVER UNRESPONSIVE
        if state.emergency_sent:
            system_status = "DRIVER UNRESPONSIVE"
        else:
            system_status = state.driver_state

        if (
            (now - state.last_driver_alert_time) >= ALERT_INTERVAL_SECS
            and not state.emergency_sent
        ):
            state.driver_warning_count  += 1
            state.last_driver_alert_time = now
            if state.driver_warning_count == 1:
                state.emergency_message = "ALERT 1 — Unsafe driver detected"
            elif state.driver_warning_count == 2:
                state.emergency_message = "ALERT 2 — Driver still unresponsive"
            elif state.driver_warning_count >= 3:
                system_status        = "DRIVER UNRESPONSIVE"
                state.emergency_sent = True
                if lat is not None: state.current_lat = lat
                if lon is not None: state.current_lon = lon
                state.emergency_message = (
                    f"ALERT 3 — EMERGENCY DISPATCHED\n"
                    f"Location: {state.current_lat}, {state.current_lon}"
                )

    else:
        state.accident_warning_count = 0
        state.driver_warning_count   = 0
        state.emergency_message      = "No Action Required"
        if high_risk_pairs:
            system_status = "PREDICTED COLLISION"
            if (now - state.last_predictive_warning_time) >= PREDICTIVE_WARNING_COOLDOWN_SECS:
                state.last_predictive_warning_time = now
                state.emergency_message = "Warning — Possible Collision Ahead"

    return {
        "system_status":     system_status,
        "driver_state":      state.driver_state,
        "emergency_message": state.emergency_message,
        "gps":               [state.current_lat, state.current_lon],
        "accident_count":    state.accident_warning_count,
        "driver_count":      state.driver_warning_count,
    }

print("process_frame defined OK")"""

'def process_frame(frame, state, lat=None, lon=None):\n    state.frame_count += 1\n    now = time.time()\n\n    if state.width is None:\n        state.height, state.width = frame.shape[:2]\n\n    HOOD_LINE_Y = int(state.height * 0.8)\n\n    # A. Driver\n    if state.frame_count % (state.fps * 10) == 0 or state.frame_count == 1:\n        state._init_driver()\n\n    # B. MiDaS\n    img_rgb     = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)\n    input_batch = midas_transform(img_rgb)\n    with torch.no_grad():\n        depth_map = midas(input_batch).squeeze().cpu().numpy()\n\n    # C. YOLO\n    results    = yolo_model(frame, conf=0.3, iou=0.4, verbose=False)[0]\n    has_masks  = results.masks is not None\n    boxes_iter = (\n        zip(results.boxes, results.masks.data) if has_masks\n        else zip(results.boxes, [None] * len(results.boxes))\n    )\n\n    raw_detections = []\n    for box, mask in boxes_iter:\n        cls  = int(box.cls[0])\n        conf = float(box.conf[0])\n        x1, y1, 

In [ ]:
def process_frame(frame, state, lat=None, lon=None):
    state.frame_count += 1
    now = time.time()

    if state.width is None:
        state.height, state.width = frame.shape[:2]

    HOOD_LINE_Y = int(state.height * 0.8)

    # A. Driver
    if state.frame_count % (state.fps * 10) == 0 or state.frame_count == 1:
        state._init_driver()

    # B. MiDaS
    img_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    input_batch = midas_transform(img_rgb)

    with torch.no_grad():
        depth_map = midas(input_batch).squeeze().cpu().numpy()

    # C. YOLO
    results = yolo_model(
        frame,
        conf=0.3,
        iou=0.4,
        verbose=False
    )[0]

    has_masks = results.masks is not None

    boxes_iter = (
        zip(results.boxes, results.masks.data)
        if has_masks
        else zip(results.boxes,[None]*len(results.boxes))
    )

    raw_detections=[]

    for box,mask in boxes_iter:

        cls=int(box.cls[0])
        conf=float(box.conf[0])

        x1,y1,x2,y2=box.xyxy[0].cpu().numpy()

        is_vehicle=cls in VEHICLE_CLASSES
        is_accident=cls in ACCIDENT_CLASSES

        if not(is_vehicle or is_accident):
            continue

        if not is_valid_vehicle_box(
            x1,y1,x2,y2,
            state.width,
            state.height
        ):
            continue

        raw_detections.append({

            "bbox":(
                float(x1),
                float(y1),
                float(x2),
                float(y2)
            ),

            "cls":cls,
            "conf":conf,

            "mask":
            mask.cpu().numpy()
            if mask is not None
            else None,

            "is_accident_class":is_accident
        })

    detections=remove_duplicates(
        raw_detections,
        iou_thresh=0.3,
        center_dist_thresh=80
    )

    tracks=state.tracker.update(
        detections,
        CLASS_NAMES
    )

    # E depth

    for t in tracks:

        if t.get("mask") is not None:

            resized_mask=cv2.resize(

                t["mask"].astype(np.uint8),

                (depth_map.shape[1],
                depth_map.shape[0]),

                interpolation=cv2.INTER_NEAREST
            )

            mask_bool=resized_mask>0

            if np.any(mask_bool):

                depth_val=float(
                    np.median(
                        depth_map[mask_bool]
                    )
                )

                state.tracker.tracks[
                    t["id"]
                ]["depth"]=depth_val

                t["depth"]=depth_val

        else:

            x1,y1,x2,y2=[
                int(v)
                for v in t["bbox"]
            ]

            cx_=int(
                np.clip(
                    (x1+x2)//2,
                    0,
                    depth_map.shape[1]-1
                )
            )

            cy_=int(
                np.clip(
                    (y1+y2)//2,
                    0,
                    depth_map.shape[0]-1
                )
            )

            depth_val=float(
                depth_map[cy_,cx_]
            )

            state.tracker.tracks[
                t["id"]
            ]["depth"]=depth_val

            t["depth"]=depth_val


    state.motion.update(tracks)

    if state.frame_count%state.fps==0 and len(tracks)>=2:

        bbox_matrix=build_future_bbox_matrix(
            tracks,
            state.motion,
            TIME_HORIZONS
        )

        state.collision_windows=(
            predict_collision_windows(
                tracks,
                bbox_matrix,
                TIME_HORIZONS,
                state.motion
            )
        )

    high_risk_pairs,\
    accident_confirmed,\
    severity_str=state.detector.update(

        tracks,
        state.motion,
        state.frame_count
    )

    system_status="SAFE"

    hood_in_frame=any(
        int(t["bbox"][3])>=HOOD_LINE_Y
        for t in tracks
    )


    # ========= التعديل المهم =========

    if state.driver_confirmed_flag["value"]:

        system_status="CONFIRMED SAFE"

        state.driver_warning_count=0
        state.accident_warning_count=0
        state.collision_frames=0

        state.detector.in_accident=False

        state.last_driver_alert_time=now
        state.last_accident_alert_time=now
        state.last_predictive_warning_time=now

        state.emergency_message="Driver confirmed OK"

        return {

            "system_status":
            system_status,

            "driver_state":
            state.driver_state,

            "emergency_message":
            state.emergency_message,

            "gps":[
                state.current_lat,
                state.current_lon
            ],

            "accident_count":0,

            "driver_count":0
        }

    # ==========================


    elif accident_confirmed or state.detector.in_accident:

        system_status="ACCIDENT DETECTED"

        if now - state.last_accident_alert_time >= ALERT_INTERVAL_SECS:

            state.accident_warning_count += 1

            state.last_accident_alert_time = now

            if state.accident_warning_count >= 3:

                state.emergency_sent = True

                state.emergency_message = (
                    "EMERGENCY DISPATCHED"
                )

    elif state.driver_state in UNSAFE_DRIVER_STATES:

        system_status = state.driver_state

        if now-state.last_driver_alert_time >= ALERT_INTERVAL_SECS:

            state.driver_warning_count += 1

            state.last_driver_alert_time = now

            if state.driver_warning_count >= 3:

                state.emergency_sent=True

                state.emergency_message=(
                    "EMERGENCY DISPATCHED"
                )

    else:

        if high_risk_pairs:

            system_status="PREDICTED COLLISION"

            state.emergency_message=(
                "Warning — Possible Collision Ahead"
            )

    return {

        "system_status":
        system_status,

        "driver_state":
        state.driver_state,

        "emergency_message":
        state.emergency_message,

        "gps":[
            state.current_lat,
            state.current_lon
        ],

        "accident_count":
        state.accident_warning_count,

        "driver_count":
        state.driver_warning_count
    }

print("process_frame updated OK")

process_frame updated OK


In [ ]:
from fastapi import FastAPI, UploadFile, Form
import nest_asyncio
import uvicorn
import threading

# Define BASE_LAT and BASE_LON to resolve NameError
BASE_LAT = 30.0444
BASE_LON = 31.2357
ALERT_INTERVAL_SECS = 5
PREDICTIVE_WARNING_COOLDOWN_SECS = 5

nest_asyncio.apply()

app   = FastAPI()
state = RoadEyeState()

@app.post("/predict")
async def predict(
    file: UploadFile,
    lat:  float = Form(None),
    lon:  float = Form(None),
):
    contents = await file.read()
    nparr    = np.frombuffer(contents, np.uint8)
    frame    = cv2.imdecode(nparr, cv2.IMREAD_COLOR)
    if frame is None:
        return {"error": "invalid image"}
    result = process_frame(frame, state, lat=lat, lon=lon)
    return result

@app.post("/confirm")
async def confirm():

    state.driver_confirmed_flag["value"] = True

    state.driver_warning_count = 0
    state.accident_warning_count = 0

    state.emergency_sent = False

    state.emergency_message = "Driver confirmed OK"

    state.detector.in_accident = False

    return {"ok": True}

# ==================== STATUS ENDPOINT (POST) ====================

@app.post("/status")
async def status(data: dict):
    state.current_lat = data.get("lat", BASE_LAT)
    state.current_lon = data.get("lon", BASE_LON)

    if state.accident_warning_count > 0:
        system_status = "AccidentDetected"
    elif len(state.collision_windows) > 0:
        system_status = "PredictedCollision"
    else:
        system_status = "Safe"

    return {
        "system_status": system_status,
        "driver_state": getattr(state, "driver_state", "SafeDriving"),
        "emergency_message": state.emergency_message,
        "frame_count": state.frame_count,
        "accident_count": state.accident_warning_count,
        "driver_count": state.driver_warning_count,
        "risk_score": calc_risk_score(),
        "lat": state.current_lat,
        "lon": state.current_lon,
        "collision_windows": len(state.collision_windows),
        "tracks_count": len(state.tracker.tracks) if hasattr(state, "tracker") else 0
    }

# ==================== STATUS ENDPOINT (GET) ← الجديد ====================

@app.get("/status")
async def status_get():

    system_status="SAFE"

    if state.detector.in_accident:

        system_status="ACCIDENT DETECTED"

    elif len(state.collision_windows)>0:

        system_status="PREDICTED COLLISION"

    elif state.driver_state in UNSAFE_DRIVER_STATES:

        system_status=state.driver_state

    return {
        "system_status": system_status,
        "driver_state": getattr(state, "driver_state", "SafeDriving"),
        "emergency_message": state.emergency_message,
        "frame_count": state.frame_count,
        "accident_count": state.accident_warning_count,
        "driver_count": state.driver_warning_count,
        "risk_score": calc_risk_score(),
        "lat": state.current_lat,
        "lon": state.current_lon,
        "collision_windows": len(state.collision_windows),
        "tracks_count": len(state.tracker.tracks) if hasattr(state, "tracker") else 0
    }

# ==================== CALC RISK SCORE ====================

def calc_risk_score():
    score = 0.0
    if state.accident_warning_count > 0:
        score = max(score, 0.95)
    if len(state.collision_windows) > 0:
        score = max(score, 0.70)
    if getattr(state, "driver_state", "") in UNSAFE_DRIVER_STATES:
        score = max(score, 0.60)
    if state.driver_warning_count > 0:
        score = max(score, 0.40)
    return round(score, 2)


nest_asyncio.apply()

def run_api():
    uvicorn.run(app, host="0.0.0.0", port=8000)

thread = threading.Thread(target=run_api, daemon=True)
thread.start()

print("✅ FastAPI started")

✅ FastAPI started


In [ ]:
from pyngrok import ngrok

# أغلقي أي tunnels قديمة
for t in ngrok.get_tunnels():
    ngrok.disconnect(t.public_url)

# افتحي tunnel جديد
public_url = ngrok.connect(8000)
print("🔗 URL للـ Flutter:", public_url)

INFO:     Started server process [27735]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


🔗 URL للـ Flutter: NgrokTunnel: "https://tricycle-maternity-margarita.ngrok-free.dev" -> "http://localhost:8000"


In [ ]:
import requests
import cv2
import time
import threading

VIDEO_PATH = "/content/drive/MyDrive/graduation project/test/w10_106.mp4"
SERVER_URL = "http://localhost:8000/predict"



FRAME_SKIP = 3

def run_video_loop(video_path, loop=True):

    print("▶ Starting video stream")

    while True:

        cap = cv2.VideoCapture(video_path)

        while True:

            ret, frame = cap.read()

            if not ret:
                break

            _, buffer = cv2.imencode(".jpg", frame)

            try:

                response = requests.post(
                    SERVER_URL,
                    files={
                        "file":(
                            "frame.jpg",
                            buffer.tobytes(),
                            "image/jpeg"
                        )
                    },
                  data={}
                )

                result=response.json()

                print(
                    f"Road:{result['system_status']} | "
                    f"Driver:{result['driver_state']}"
                )

            except Exception as e:
                print(e)

            time.sleep(0.1)

        cap.release()

        if not loop:
            break

video_thread=threading.Thread(
    target=run_video_loop,
    args=(VIDEO_PATH,),
    daemon=True
)

video_thread.start()

print("✅ Video loop running")

▶ Starting video stream
✅ Video loop running


In [ ]:
from pyngrok import ngrok

ngrok.kill()
print("Old ngrok closed")

Road:CONFIRMED SAFE | Driver:DangerousDriving
Old ngrok closed


In [ ]:
!ngrok config add-authtoken 3FkaisKhSFTLPTLM2GszCBMB7BC_2Bf8Ka8pNEQCjfqJ65jRq


Authtoken saved to configuration file: /root/.config/ngrok/ngrok.yml


In [ ]:
from pyngrok import ngrok

for tunnel in ngrok.get_tunnels():
    ngrok.disconnect(tunnel.public_url)

public_url = ngrok.connect(8000)

print(public_url)

ACCIDENT CONFIRMED at frame 1390 — MINOR
INFO:     127.0.0.1:43388 - "POST /predict HTTP/1.1" 200 OK
Road:CONFIRMED SAFE | Driver:DangerousDriving
ACCIDENT CONFIRMED at frame 1391 — SEVERE
INFO:     127.0.0.1:43390 - "POST /predict HTTP/1.1" 200 OK
Road:CONFIRMED SAFE | Driver:DangerousDriving
NgrokTunnel: "https://tricycle-maternity-margarita.ngrok-free.dev" -> "http://localhost:8000"
